# Prédiction du Churn Bancaire — XGBoost + Seuil Optimal + SHAP

**Auteur :** Emmanuel TSAGUE — EDF MAD EDVANCE | Datascientest 2024  
**Données :** simulées — 15 000 clients, churn ~16%  

> *Acquérir un client coûte 5 à 25× plus cher que le retenir.*  
> Identifier les churners à l'avance permet des actions de rétention ciblées.

### Plan
1. Exploration & déséquilibre des classes
2. Baseline naïve — le piège de l'accuracy
3. XGBoost avec scale_pos_weight
4. Optimisation du seuil (F1.5 — priorité recall)
5. Évaluation coût métier
6. SHAP — profil du client à risque
7. Segments prioritaires pour la rétention

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

from src.data_generation import load_or_generate
from src.churn_model import (encode_categoricals, build_xgb_pipeline,
                              find_optimal_threshold, evaluate_churn_model,
                              plot_shap_churn)
print('Imports OK')

## 1. Exploration — déséquilibre des classes

In [ ]:
df = load_or_generate('../data_sample/bank_churn_simulated.csv')
print(f'Clients : {len(df):,} | Taux churn : {df.churn.mean():.1%}')
print(df.value_counts('churn'))
df.head()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
cols = ['credit_score','age','anciennete_ans','solde','nb_produits','membre_actif','salaire_annuel']
for ax, col in zip(axes.flat, cols):
    df[df.churn==0][col].hist(ax=ax, alpha=0.6, bins=30, label='Reste',  color='#4CAF50')
    df[df.churn==1][col].hist(ax=ax, alpha=0.7, bins=30, label='Churn',  color='#F44336')
    ax.set_title(col); ax.legend(fontsize=8)
plt.suptitle('Profils clients : churners vs fidèles', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Le piège de l'accuracy

> Un modèle qui dit *toujours 'reste'* a **84% d'accuracy** mais **0% de recall**.  
> → Toujours évaluer le recall et le F1 sur la classe minoritaire.

In [ ]:
df_enc = encode_categoricals(df)
X = df_enc.drop(columns='churn'); y = df_enc['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_naive = np.zeros(len(y_test))
print(f'Modèle naïf (tout=0) :')
print(f'  Accuracy = {(y_naive == y_test.values).mean():.1%}  ← trompeur !')
print(f'  Recall   = 0.00  (détecte 0 churner sur {y_test.sum()})')

## 3. XGBoost avec scale_pos_weight

> `scale_pos_weight = n_négatifs / n_positifs` : donne plus de poids aux churners pendant l'entraînement.

In [ ]:
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {spw:.2f}')

pipe = build_xgb_pipeline(scale_pos_weight=spw)
pipe.fit(X_train, y_train)
y_proba = pipe.predict_proba(X_test)[:, 1]
print(f'AUC = {roc_auc_score(y_test, y_proba):.4f}')

## 4. Optimisation du seuil

> Le seuil par défaut est 0.5. En churn, on préfère un **recall élevé** (ne pas rater de churners)  
> quitte à avoir quelques fausses alarmes → F1.5 (β=1.5 donne plus de poids au recall).

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Courbe PR
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=ax1, name='XGBoost')
ax1.set_title('Courbe Précision-Rappel')

# Seuil optimal
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
f15 = 3.25 * precision * recall / (2.25 * precision + recall + 1e-9)
ax2.plot(thresholds, f15[:-1], color='#2196F3', label='F1.5')
ax2.axvline(thresholds[f15[:-1].argmax()], color='red', linestyle='--',
            label=f'Seuil opt={thresholds[f15[:-1].argmax()]:.3f}')
ax2.set_title('Score F1.5 par seuil'); ax2.set_xlabel('Seuil'); ax2.legend()

plt.tight_layout(); plt.show()

## 5. Évaluation avec coût métier

| Erreur | Description | Coût |
|--------|-------------|------|
| FP | Client non-churner contacté inutilement | 50 € |
| FN | Churner manqué (client perdu) | 500 € |

In [ ]:
metrics = evaluate_churn_model(pipe, X_test, y_test, cost_retention=50, cost_lost_client=500)

## 6. SHAP — Profil du client à risque

> **Objectif :** comprendre quelles variables font qu'un client va churner.  
> Permet aux conseillers de personnaliser l'action de rétention.

In [ ]:
# Récupérer le modèle XGBoost du pipeline
try:
    model_step = pipe.named_steps['model']
    plot_shap_churn(model_step, X_test, n_samples=500)
except Exception as e:
    print(f'SHAP non disponible : {e}')

## 7. Segments prioritaires

> Identifier les segments avec le plus fort taux de churn → prioriser les actions.

In [ ]:
df_test = X_test.copy()
df_test['churn_reel']   = y_test.values
df_test['churn_predit'] = (y_proba >= metrics['best_threshold']).astype(int)
df_test['proba_churn']  = y_proba
df_test['pays_orig']    = df.loc[X_test.index, 'pays'].values

print('Taux de churn prédit par pays :')
for pays, grp in df_test.groupby('pays_orig'):
    taux = grp['proba_churn'].mean()
    n    = len(grp)
    print(f'  {pays:<12} : {taux:.1%} (n={n})')

## Synthèse

| Étape | Outil | Résultat |
|-------|-------|----------|
| Déséquilibre détecté | value_counts, baseline naïve | Accuracy = piège |
| XGBoost équilibré | scale_pos_weight | AUC ~0.87 |
| Seuil optimal | F1.5 (β=1.5) | Recall maximisé |
| Coût métier | FP×50€ + FN×500€ | Décision opérationnelle |
| SHAP global | beeswarm | Top variables : membre_actif, nb_produits |
| Segmentation | groupby + proba | Pays / profil à cibler |

*Données simulées (seed=42) — aucune donnée bancaire réelle — Emmanuel TSAGUE*